# 01 — SMILES & Molecule Exploration

**SPORA — Hyphae Composites**

This notebook is your sandbox for exploring polymer molecules with RDKit before building them into the pipeline.
Nothing you do here affects the database or the pipeline code — it is a safe place to experiment.

---

### What is SMILES?
SMILES (Simplified Molecular Input Line Entry System) is a way of writing a molecule as a text string.
For example, acetic acid (vinegar) is `CC(=O)O`. RDKit can take that string and turn it into a full molecular object you can visualise, measure, and react.

Run each cell with **Shift + Enter**.

In [ ]:
# First, make sure RDKit is available
from rdkit import Chem
from rdkit.Chem import Draw, Descriptors, rdMolDescriptors
from rdkit.Chem.Draw import IPythonConsole
import pandas as pd

print('RDKit imported successfully ✓')

## 1. Load a simple molecule

In [ ]:
# Load PLA's monomer unit from its SMILES string
# This is the same SMILES stored in the polymers table in Supabase
pla_smiles = 'CC(OC(=O))n'
mol = Chem.MolFromSmiles(pla_smiles)

# Draw it
mol

## 2. Compute descriptors for the PLA monomer

In [ ]:
# These are the same descriptors SPORA stores in the descriptors table
descriptors = {
    'Molecular Weight':     Descriptors.MolWt(mol),
    'LogP':                 Descriptors.MolLogP(mol),
    'TPSA':                 Descriptors.TPSA(mol),
    'H-bond Donors':        rdMolDescriptors.CalcNumHBD(mol),
    'H-bond Acceptors':     rdMolDescriptors.CalcNumHBA(mol),
    'Rotatable Bonds':      rdMolDescriptors.CalcNumRotatableBonds(mol),
    'Ring Count':           rdMolDescriptors.CalcNumRings(mol),
}

pd.DataFrame(descriptors, index=['PLA monomer']).T

## 3. Compare all five SPORA polymers

In [ ]:
# Load all five supported polymers and compare their descriptors
# These SMILES match what was seeded into the Supabase polymers table
polymers = {
    'PLA':  'CC(OC(=O))n',
    'PETG': 'OCCOC(=O)c1ccc(cc1)C(=O)O',
    'PP':   'CC(C)C',
    'HDPE': 'CC',
}

rows = []
for name, smiles in polymers.items():
    m = Chem.MolFromSmiles(smiles)
    if m is None:
        print(f'Warning: could not parse {name} — {smiles}')
        continue
    rows.append({
        'Polymer':      name,
        'Mol Weight':   round(Descriptors.MolWt(m), 2),
        'LogP':         round(Descriptors.MolLogP(m), 2),
        'TPSA':         round(Descriptors.TPSA(m), 2),
        'HBD':          rdMolDescriptors.CalcNumHBD(m),
        'HBA':          rdMolDescriptors.CalcNumHBA(m),
    })

pd.DataFrame(rows).set_index('Polymer')

## 4. Draw all polymers side by side

In [ ]:
mols = [Chem.MolFromSmiles(s) for s in polymers.values()]
Draw.MolsToGridImage(mols, molsPerRow=2, subImgSize=(400, 300), legends=list(polymers.keys()))

## 5. Your turn — try it with a new molecule

Change the SMILES string below to any molecule you want to explore.
You can look up SMILES strings on [PubChem](https://pubchem.ncbi.nlm.nih.gov/) — search for any molecule and find the SMILES under "Canonical SMILES".

In [ ]:
# Try your own molecule here
my_smiles = 'CC(=O)O'  # ← replace this with any SMILES you find on PubChem

my_mol = Chem.MolFromSmiles(my_smiles)
if my_mol is None:
    print('Could not parse that SMILES — double-check the string')
else:
    print(f'Molecular weight: {Descriptors.MolWt(my_mol):.2f} Da')
    display(my_mol)

---
**Next:** Once you are comfortable with SMILES and descriptors, move on to implementing `build_polymer_smiles()` in `spora/rdkit_pipeline/smiles_builder.py`. The goal is to chain `n_repeat` copies of the monomer into an oligomer SMILES.